In [4]:
!pip install aiohttp nest_asyncio pandas tqdm > /dev/null

import requests, re, aiohttp, asyncio, nest_asyncio, pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

nest_asyncio.apply()

base_url = "https://www.promofarma.com.br"
categoria_url = f"{base_url}/medicamentos"
headers = {"User-Agent": "Mozilla/5.0"}

# ============================
# PASSO 1: Coletar links de produtos
# ============================
def coletar_links_categoria(url, max_paginas=1, limit=10):
    links = []
    for page in range(1, max_paginas+1):
        print(f"🔎 Coletando página {page}...")
        resp = requests.get(f"{url}?page={page}", headers=headers)
        if resp.status_code != 200:
            print("❌ Erro HTTP", resp.status_code)
            break
        soup = BeautifulSoup(resp.text, "html.parser")
        page_links = [a["href"] for a in soup.select("a[href*='/p']")]
        links.extend(page_links)
        if len(links) >= limit:
            break
    return list(set(links[:limit]))

produto_links = coletar_links_categoria(categoria_url, max_paginas=2, limit=10)
print(f"✅ Encontrados {len(produto_links)} produtos (limitados)")

# ============================
# PASSO 2: Extrair productId real de cada página
# ============================
def extrair_product_id(produto_url):
    resp = requests.get(produto_url, headers=headers)
    if resp.status_code != 200:
        return None
    match = re.search(r'"productId":"(\d+)"', resp.text)
    if match:
        return match.group(1)
    return None

product_ids = []
for link in produto_links:
    url = base_url + link if link.startswith("/") else link
    pid = extrair_product_id(url)
    if pid:
        product_ids.append(pid)

print("🔑 IDs encontrados:", product_ids)

# ============================
# PASSO 3: Montar URLs da API
# ============================
api_urls = [f"{base_url}/api/catalog_system/pub/products/search?fq=productId:{pid}" for pid in product_ids]
print("Exemplo de URL da API:", api_urls[0] if api_urls else "Nenhuma")

# ============================
# PASSO 4: Baixar JSONs em paralelo
# ============================
async def fetch(session, url):
    try:
        async with session.get(url, headers=headers) as resp:
            if resp.status != 200:
                return None
            return await resp.json()
    except:
        return None

async def baixar_produtos(api_urls):
    produtos = []
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, url) for url in api_urls]
        for f in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            result = await f
            if result:
                produtos.extend(result)
    return produtos

produtos = asyncio.run(baixar_produtos(api_urls))
print(f"📦 Total de produtos coletados: {len(produtos)}")

# ============================
# PASSO 5: Transformar em CSV
# ============================
dados = []
for p in produtos:
    try:
        item = p["items"][0]
        oferta = item["sellers"][0]["commertialOffer"]
        dados.append({
            "id": p["productId"],
            "nome": p["productName"],
            "marca": p.get("brand", ""),
            "preco": oferta["Price"],
            "preco_de": oferta["ListPrice"],
            "estoque": oferta["AvailableQuantity"],
            "ean": item.get("ean", ""),
            "link": base_url + p["link"],
            "imagem": item["images"][0]["imageUrl"] if item["images"] else ""
        })
    except:
        continue

df = pd.DataFrame(dados)
df.to_csv("produtos_limitados.csv", index=False, encoding="utf-8-sig")
df.head()


🔎 Coletando página 1...
✅ Encontrados 9 produtos (limitados)
🔑 IDs encontrados: ['4893', '4735', '4205', '2032', '4623', '2360', '1881', '4727']
Exemplo de URL da API: https://www.promofarma.com.br/api/catalog_system/pub/products/search?fq=productId:4893


100%|██████████| 8/8 [00:00<00:00, 19.63it/s]

📦 Total de produtos coletados: 8


,id,nome,marca,preco,preco_de,estoque,ean,link,imagem
0,2032,Acetilcisteína 600mg Granulado 16 Envelopes De...,Genérico,23.25,76.40,99999,7896004713342,https://www.promofarma.com.brhttps://www.promo...,https://promofarma.vteximg.com.br/arquivos/ids...
1,4727,Simeticona 125mg 10 Cápsulas Moles Medley,Genérico,9.21,10.90,99999,7896422508742,https://www.promofarma.com.brhttps://www.promo...,https://promofarma.vteximg.com.br/arquivos/ids...
2,4623,Rosuvastatina Cálcica 10mg 30 Comprimidos Reve...,Genérico,11.69,58.02,99999,7896422528580,https://www.promofarma.com.brhttps://www.promo...,https://promofarma.vteximg.com.br/arquivos/ids...
3,4205,Orlistate 120mg 84 Cápsulas Gelatinosas Duras Ems,Genérico,102.65,404.45,1,7896004737225,https://www.promofarma.com.brhttps://www.promo...,https://promofarma.vteximg.com.br/arquivos/ids...
4,2360,Esomeprazol Magnésico 40mg 28 Comprimidos Reve...,Genérico,42.36,281.45,99999,7897076914088,https://www.promofarma.com.brhttps://www.promo...,https://promofarma.vteximg.com.br/arquivos/ids...
